# Figure 3 and Figure 5 analyses

This notebook contains the reproducible data-processing and plotting workflows used for
Figures 3 and 5.

## Scope

- Figure 3b: alternative-splicing event composition
- Figure 3e: tumor-specific transcript identification
- Figure 5b: isoform-ratio correlation analysis
- Figure 5c: known and novel transcript counts
- Figure 5e: direction and pattern classification of significant transcripts
- Figure 5f: preparation of dot-plot input data

All paths and analysis parameters are defined in the configuration section. Notebook
outputs have been cleared before release.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="white")


## Configuration

Update `project_dir` when running the notebook in a different environment.
Input files are expected to retain the same column structure as the original analysis.


In [ ]:
project_dir = Path("/data1/DYY/bambu")
output_dir = project_dir / "plot" / "figures_3_and_5"
output_dir.mkdir(parents=True, exist_ok=True)

group_file = project_dir / "288_group.csv"
transcript_cpm_file = project_dir / "CPM_transcript.txt"
normalized_count_file = (
    project_dir / "288_count_matrix_deseq2" / "all_trans_norm.counts.fn"
)
adjusted_pvalue_file = project_dir / "288_mstrg_tpm" / "all_trans_padj.csv"
isoform_ratio_dir = project_dir / "288_mstrg_tpm"
suppa_event_dir = project_dir / "288_normal_trans_suppa_types"
deseq_result_dir = project_dir / "288_desgene_df"

significance_cutoff = 0.05
minimum_expression = 0.05


## Shared utility functions


In [ ]:
def require_file(file_path):
    """Raise a clear error when a required input file is unavailable."""
    if not file_path.is_file():
        raise FileNotFoundError(f"Required file was not found: {file_path}")


def require_columns(table, required_columns, table_name):
    """Validate that a table contains all required columns."""
    missing_columns = sorted(set(required_columns) - set(table.columns))
    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: {missing_columns}"
        )


def read_table(file_path, **kwargs):
    """Read a tab-delimited table after validating the file path."""
    require_file(file_path)
    return pd.read_csv(file_path, sep="\t", **kwargs)


def save_figure(figure, file_name, dpi=400):
    """Save a figure using consistent publication settings."""
    figure.savefig(
        output_dir / file_name,
        dpi=dpi,
        bbox_inches="tight",
    )


## Load and validate shared metadata


In [ ]:
sample_metadata = read_table(group_file)
require_columns(
    sample_metadata,
    {"name", "Sample ID", "Type"},
    "Sample metadata",
)

sample_metadata = sample_metadata.copy()
sample_metadata["tissue"] = sample_metadata["Sample ID"].str[:3]

if sample_metadata["name"].duplicated().any():
    duplicated_names = sample_metadata.loc[
        sample_metadata["name"].duplicated(), "name"
    ].tolist()
    raise ValueError(f"Duplicated sample names were found: {duplicated_names[:10]}")

sample_metadata.head()


# Figure 3b: alternative-splicing event composition

The original notebook combined SUPPA event-class files across tissues and summarized
the relative contribution of each event type. The cleaned workflow explicitly aligns
event identifiers and tissue names.


In [ ]:
def load_suppa_event_matrix(event_dir, tissues):
    """Create a binary transcript-by-tissue matrix from SUPPA event files."""
    event_files = sorted(event_dir.glob("*"))
    if not event_files:
        raise FileNotFoundError(f"No SUPPA event files were found in {event_dir}")

    event_matrix = pd.DataFrame()

    for event_file in event_files:
        tissue = event_file.name[:3]
        if tissue not in tissues:
            continue

        event_table = pd.read_csv(event_file, sep="\t", index_col=0)
        event_ids = event_table.index.astype(str)

        if tissue not in event_matrix.columns:
            event_matrix[tissue] = 0

        event_matrix.loc[event_ids, tissue] = 1

    return event_matrix.fillna(0).astype(int)


tissue_order = sample_metadata["tissue"].drop_duplicates().tolist()
suppa_event_matrix = load_suppa_event_matrix(suppa_event_dir, tissue_order)


In [ ]:
event_type_counts = {}

for event_file in sorted(suppa_event_dir.glob("*")):
    tissue = event_file.name[:3]
    event_type = event_file.name[:2]

    if tissue not in tissue_order:
        continue

    event_table = pd.read_csv(event_file, sep="\t", index_col=0)
    event_type_counts.setdefault(tissue, {})
    event_type_counts[tissue][event_type] = event_table.shape[0]

event_count_table = (
    pd.DataFrame(event_type_counts)
    .fillna(0)
    .astype(int)
    .reindex(columns=tissue_order)
)

event_proportion_table = event_count_table.div(
    event_count_table.sum(axis=0).replace(0, np.nan),
    axis=1,
)

figure, axis = plt.subplots(figsize=(6, 4))
event_proportion_table.T.plot(
    kind="bar",
    stacked=True,
    ax=axis,
    colormap="viridis",
)
axis.set_xlabel("")
axis.set_ylabel("Proportion of splicing events")
axis.legend(
    title="Event type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)
sns.despine(ax=axis)
save_figure(figure, "figure_3b_splicing_event_composition.pdf")
plt.show()


# Figure 3e: tumor-specific transcript identification

A transcript is considered absent from normal samples when its normalized expression is
below `minimum_expression` in every normal sample. Among those transcripts, candidates
expressed above the threshold in at least one tumor sample are retained.


In [ ]:
normalized_counts = read_table(normalized_count_file, index_col=0).T

expression_with_metadata = (
    normalized_counts
    .reset_index(names="name")
    .merge(
        sample_metadata[["name", "Type"]],
        on="name",
        how="inner",
        validate="one_to_one",
    )
)

normal_expression = (
    expression_with_metadata.loc[
        expression_with_metadata["Type"].eq("Normal")
    ]
    .drop(columns="Type")
    .set_index("name")
)

tumor_expression = (
    expression_with_metadata.loc[
        expression_with_metadata["Type"].eq("Tumor")
    ]
    .drop(columns="Type")
    .set_index("name")
)

normal_absent_transcripts = normal_expression.columns[
    normal_expression.lt(minimum_expression).all(axis=0)
]

tumor_specific_transcripts = tumor_expression.loc[
    :,
    normal_absent_transcripts,
].columns[
    tumor_expression.loc[:, normal_absent_transcripts]
    .ge(minimum_expression)
    .any(axis=0)
]

transcript_annotation = read_table(
    transcript_cpm_file,
    usecols=["TXNAME", "GENEID"],
)

tumor_specific_gene_ids = (
    pd.DataFrame({"TXNAME": tumor_specific_transcripts})
    .merge(transcript_annotation, on="TXNAME", how="left")
    .dropna(subset=["GENEID"])
)

tumor_specific_gene_ids["GENEID"] = (
    tumor_specific_gene_ids["GENEID"]
    .str.split(".")
    .str[0]
)

tumor_specific_gene_ids.to_csv(
    output_dir / "figure_3e_tumor_specific_gene_ids.tsv",
    sep="\t",
    index=False,
)


# Figure 5b: isoform-ratio correlation

Isoform-ratio matrices are combined across tissues, aligned to paired normal and tumor
samples, and used to calculate a sample-to-sample Pearson correlation matrix.


In [ ]:
adjusted_pvalues = read_table(adjusted_pvalue_file, index_col=0)
significant_transcripts = adjusted_pvalues.index[
    adjusted_pvalues.lt(significance_cutoff).any(axis=1)
]


def load_isoform_ratio_tables(input_dir):
    """Load sample-level isoform-ratio tables from the analysis directory."""
    ratio_tables = []

    for file_path in sorted(input_dir.iterdir()):
        if not file_path.is_file() or len(file_path.name) != 7:
            continue

        tissue = file_path.name[:3]
        ratio_table = pd.read_csv(file_path, sep="\t", index_col=0).T
        ratio_table["tissue"] = tissue
        ratio_tables.append(ratio_table)

    if not ratio_tables:
        raise FileNotFoundError(
            f"No isoform-ratio tables matching the expected naming rule were found "
            f"in {input_dir}"
        )

    return pd.concat(ratio_tables, axis=0)


isoform_ratios = load_isoform_ratio_tables(isoform_ratio_dir)
available_significant_transcripts = isoform_ratios.columns.intersection(
    significant_transcripts
)
isoform_ratios = isoform_ratios.loc[:, available_significant_transcripts]


In [ ]:
ratio_with_metadata = (
    isoform_ratios
    .reset_index(names="name")
    .merge(sample_metadata, on="name", how="inner")
    .sort_values(["Sample ID", "Type"])
)

normal_ratios = (
    ratio_with_metadata.loc[ratio_with_metadata["Type"].eq("Normal")]
    .set_index("Sample ID")
    .drop(columns=["name", "Group", "Type", "tissue"], errors="ignore")
)

tumor_ratios = (
    ratio_with_metadata.loc[ratio_with_metadata["Type"].eq("Tumor")]
    .set_index("Sample ID")
    .drop(columns=["name", "Group", "Type", "tissue"], errors="ignore")
)

paired_sample_ids = normal_ratios.index.intersection(tumor_ratios.index)
normal_ratios = normal_ratios.loc[paired_sample_ids].astype(float)
tumor_ratios = tumor_ratios.loc[paired_sample_ids].astype(float)

normal_for_correlation = normal_ratios.T.copy()
tumor_for_correlation = tumor_ratios.T.copy()

normal_for_correlation.columns = (
    normal_for_correlation.columns.astype(str) + "_N"
)
tumor_for_correlation.columns = (
    tumor_for_correlation.columns.astype(str) + "_T"
)

ratio_correlation = pd.concat(
    [normal_for_correlation, tumor_for_correlation],
    axis=1,
).corr(method="pearson")


In [ ]:
figure, axis = plt.subplots(figsize=(9, 8))
sns.heatmap(
    ratio_correlation,
    cmap="vlag",
    center=0,
    square=True,
    xticklabels=False,
    yticklabels=False,
    ax=axis,
)
axis.set_title("Isoform-ratio correlation")
save_figure(figure, "figure_5b_isoform_ratio_correlation.pdf")
plt.show()

ratio_correlation.to_csv(
    output_dir / "figure_5b_isoform_ratio_correlation.tsv",
    sep="\t",
)


# Figure 5c: known and novel significant transcripts

Significant transcripts are classified as known when their identifiers begin with
`ENST`; all remaining identifiers are classified as novel.


In [ ]:
significant_transcript_table = (
    adjusted_pvalues.loc[significant_transcripts]
    .reset_index()
)

transcript_id_column = significant_transcript_table.columns[0]
significant_transcript_table = significant_transcript_table.rename(
    columns={transcript_id_column: "transcript_id"}
)

transcript_class_counts = pd.DataFrame(
    {
        "known": [
            significant_transcript_table["transcript_id"]
            .astype(str)
            .str.startswith("ENST")
            .sum()
        ],
        "novel": [
            (~significant_transcript_table["transcript_id"]
             .astype(str)
             .str.startswith("ENST"))
            .sum()
        ],
    },
    index=["All tissues"],
)

figure, axis = plt.subplots(figsize=(3.5, 3.5))
transcript_class_counts.plot(
    kind="bar",
    stacked=True,
    ax=axis,
)
axis.set_xlabel("")
axis.set_ylabel("Number of significant transcripts")
axis.legend(title="", frameon=False)
axis.tick_params(axis="x", rotation=0)
sns.despine(ax=axis)
save_figure(figure, "figure_5c_known_and_novel_transcripts.pdf")
plt.show()

transcript_class_counts.to_csv(
    output_dir / "figure_5c_known_and_novel_transcripts.tsv",
    sep="\t",
)


# Figure 5e: direction and cross-tissue pattern classification

For each tissue, significant transcripts are classified by the sign of their effect
estimate and by whether their tissue-level pattern is convergent or scattered.
The exact column names in each differential-expression file may differ; update
`effect_column` and `pattern_column` below when necessary.


In [ ]:
effect_column = "log2FoldChange"
pattern_column = "pattern"

direction_counts = {}
pattern_counts = {}

for tissue in adjusted_pvalues.columns:
    result_file = deseq_result_dir / f"{tissue}.csv"
    if not result_file.is_file():
        continue

    tissue_results = pd.read_csv(result_file, sep="\t", index_col=0)
    tissue_results = tissue_results.loc[
        tissue_results.index.intersection(significant_transcripts)
    ]

    if effect_column in tissue_results.columns:
        direction_counts[tissue] = {
            "up": tissue_results[effect_column].gt(0).sum(),
            "down": tissue_results[effect_column].lt(0).sum(),
            "other": tissue_results[effect_column].eq(0).sum(),
        }

    if pattern_column in tissue_results.columns:
        normalized_pattern = tissue_results[pattern_column].astype(str).str.lower()
        pattern_counts[tissue] = {
            "convergent": normalized_pattern.eq("con").sum(),
            "scattered": normalized_pattern.eq("sca").sum(),
            "other": (~normalized_pattern.isin(["con", "sca"])).sum(),
        }

direction_count_table = pd.DataFrame(direction_counts).fillna(0).astype(int)
pattern_count_table = pd.DataFrame(pattern_counts).fillna(0).astype(int)


In [ ]:
if not direction_count_table.empty:
    direction_proportions = direction_count_table.div(
        direction_count_table.sum(axis=0).replace(0, np.nan),
        axis=1,
    )

    figure, axis = plt.subplots(figsize=(5, 3))
    direction_proportions.T.plot(
        kind="bar",
        stacked=True,
        ax=axis,
    )
    axis.set_xlabel("")
    axis.set_ylabel("Transcript proportion")
    axis.legend(title="", frameon=False)
    sns.despine(ax=axis)
    save_figure(figure, "figure_5e_direction_classification.pdf")
    plt.show()

if not pattern_count_table.empty:
    pattern_proportions = pattern_count_table.div(
        pattern_count_table.sum(axis=0).replace(0, np.nan),
        axis=1,
    )

    figure, axis = plt.subplots(figsize=(5, 3))
    pattern_proportions.T.plot(
        kind="bar",
        stacked=True,
        ax=axis,
    )
    axis.set_xlabel("")
    axis.set_ylabel("Transcript proportion")
    axis.legend(title="", frameon=False)
    sns.despine(ax=axis)
    save_figure(figure, "figure_5e_pattern_classification.pdf")
    plt.show()


# Figure 5f: dot-plot input preparation

The output table contains adjusted P values for transcripts significant in multiple
tissues. Transcript identifiers are retained explicitly so that gene annotation can be
joined in a separate, traceable step.


In [ ]:
minimum_significant_tissues = 3

multi_tissue_transcripts = adjusted_pvalues.index[
    adjusted_pvalues.lt(significance_cutoff).sum(axis=1)
    >= minimum_significant_tissues
]

figure_5f_pvalues = adjusted_pvalues.loc[
    multi_tissue_transcripts
].copy()

figure_5f_pvalues.index.name = "transcript_id"
figure_5f_pvalues.to_csv(
    output_dir / "figure_5f_adjusted_pvalues.tsv",
    sep="\t",
)

negative_log10_pvalues = -np.log10(
    figure_5f_pvalues.clip(lower=np.finfo(float).tiny)
)
negative_log10_pvalues.to_csv(
    output_dir / "figure_5f_negative_log10_pvalues.tsv",
    sep="\t",
)

negative_log10_pvalues.head()


## Reproducibility notes

- All notebook outputs and execution counters were cleared before release.
- Input paths and thresholds are defined in one configuration section.
- Sample and transcript alignment is performed using explicit identifiers.
- Figure 2, Figure 6, exploratory t-SNE analyses, and temporary debugging cells from the
  original notebook are intentionally excluded because they are outside this notebook's
  stated scope.
